# Independent cash rebalance and action accounting

Can a reader reconstruct the first rebalance and trace dollars to ending NAV?

This notebook reviews saved retrospective evidence. New descriptive calculations are post-study analysis, not new strategy selection or an untouched test.

In [1]:
from pathlib import Path
import json,sys,math,statistics
r=Path.cwd()
if not (r/'artifacts/research').exists(): r=r.parent
sys.path.insert(0,str(r/'src'));sys.path.insert(0,str(r))
from scripts.review_visuals import table,figure,lines,bars
if 'display_bundle' not in globals():
 from IPython.display import display
 def display_bundle(data): display(data,raw=True)
b=json.loads((r/'artifacts/research/public_bundle.json').read_text());a=b['analysis']
p=json.loads((r/'artifacts/review/polish.json').read_text());chosen=b['selected']
print('Saved test:',b['as_of'],'Selected:',chosen,'Post-study review:',p['review_date'])

Saved test: 2025-12-31 Selected: stumps_24 Post-study review: 2026-11-25


## Whole-share economic build

The independent loop sizes in ticker order and applies fees/slippage exactly once.

In [2]:
v=b['review'];cash=v['initial_nav'];mv=0;rows=[]
for x in v['assets']:
 px=x['raw_open']*(1+v['slippage_bps']/1e4);q=min(math.floor(v['initial_nav']*x['target_weight']/x['raw_open']),math.floor(x['volume']*v['participation']),math.floor(max(cash,0)/(px*(1+v['fee_bps']/1e4))))
 fee=q*px*v['fee_bps']/1e4;cash-=q*px+fee;mv+=q*x['raw_close'];rows.append([x['asset'],q,round(fee,4),round(cash,4)])
assert abs(cash+mv-v['python_ledger']['nav'])<1e-7;display_bundle(table(['Fund','Shares','Fee','Cash after'],rows));print('Closing NAV:',cash+mv)

Fund,Shares,Fee,Cash after
XLB,0,0.0,100000.0
XLE,383,3.3307,66689.6641
XLF,0,0.0,66689.6641
XLI,0,0.0,66689.6641
XLK,265,3.3343,33343.2666
XLP,0,0.0,33343.2666
XLU,0,0.0,33343.2666
XLV,0,0.0,33343.2666
XLY,256,3.3276,63.7892


Closing NAV: 97918.169238186


## Corporate-action test controls

Separate ex-date entitlement, payable-date cash and splits. Fictional fixtures do not provide reported market returns.

In [3]:
import importlib.util
spec=importlib.util.spec_from_file_location('economics',r/'tests/test_economics.py');t=importlib.util.module_from_spec(spec);spec.loader.exec_module(t)
t.test_hand_calculated_split_dividend_receivable_and_payment_preserve_nav();t.test_new_ex_dividend_buyer_does_not_receive_old_holders_distribution();print('Independent split, entitlement and payment checks: pass')

Independent split, entitlement and payment checks: pass


## Full saved-dollar reconciliation

Fund contribution is a self-financing P&L identity, not a causal skill decomposition.

In [4]:
display_bundle(table(['Policy','Sum fund P&L','Ending NAV less capital','Daily max residual'],[[n,round(sum(x['cumulative_net_pnl_by_asset'].values()),4),round(b['results'][n]['metrics']['final_nav']-100000,4),x['attribution_max_daily_residual']] for n,x in a['strategies'].items()]));assert all(abs(x['attribution_total_difference'])<1e-6 for x in a['strategies'].values())

Policy,Sum fund P&L,Ending NAV less capital,Daily max residual
equal_weight,48455.1325,48455.1325,4.547473508864641e-11
momentum_63,44751.1849,44751.1849,2.9103830456733704e-11
inverse_vol,41642.2314,41642.2314,2.7284841053187847e-11
stumps_24,41788.8364,41788.8364,3.956301952712238e-11
SPY_buy_hold,82757.8281,82757.8281,2.9103830456733704e-11


## Interpretation

The audit sample reconciles independently. The editable Excel lab is a first-rebalance scenario, not a separate full-history ML implementation. Daily volume and an opening mark do not establish auction execution.